In [0]:
# Retrieve values from previous task (bronze_output)
bronze_output = dbutils.jobs.taskValues.get(taskKey="Bronze",key="bronze_output")
silver_data = dbutils.jobs.taskValues.get(taskKey="Silver",key="silver_output")

# get each variable
start_date = bronze_output.get("start_date","")
silver_adls = bronze_output.get("silver_adls","")
gold_adls = bronze_output.get("gold_adls","")

print(f"start_date : {start_date} , Gold_adls:{gold_adls}")

In [0]:
'''
reverse_geocoder is particularly useful in your earthquake API project because the USGS earthquake data gives you latitude and longitude, but you may want to know the nearest city/country for each earthquake.
'''

%pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268113 sha256=884355e848fb38000fcdc9b13ab4a0ffcd150ce21a18f04ab1e4c098fa8d5dc7
  Stored in directory: /home/spark-7ec0fc60-dd9d-44c9-b676-47/.cache/pip/wheels/11/e1/67/6e47f0ad41ea1843d37e1fbe79c6074744a1f4aace641cf800
Successfully built reverse_geocoder
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType

# Ensure the below library is installed on your cluster
import reverse_geocoder as rg
from datetime import date, timedelta

In [0]:
# we need only a day data from parquet file, as data is appended in silver layer.
df = spark.read.parquet(silver_data).filter(col('time') > start_date)

''' 
#RUN THIS CELL ONLY TO RUN THIS NOTEBOOK SEPERATLY AND NOT PART OF WORKFLOW | PIPELINE, provide silver file path
from datetime import date, timedelta
start_date = date.today() - timedelta(1)
df = spark.read.parquet("abfss://silver@databricksstore88.dfs.core.windows.net/earthquake_events_silver/earthquake_events_silver").filter(col('time') > start_date)
'''


In [0]:
df = df.limit(10)

In [0]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    try:
        coordinates = (float(lat), float(lon))
        result = rg.search(coordinates)[0].get('cc')
        print(f"Processed coordinates: {coordinates} -> {result}")
        return result
    except Exception as e:
        print(f"Error processing coordinates: {lat}, {lon} -> {str(e)}")
        return None

In [0]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

In [0]:
# adding country_code and city attributes
df_with_location = df.withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

In [0]:
# adding significance classification
df_with_location_sig_class = df_with_location.withColumn('sig_class',
                                                         when(col("sig")< 100,"Low").
                                                         when((col("sig") >=100) & (col("sig") < 500),"Moderate").
                                                         otherwise("High"))

In [0]:
# Save the transformed DataFrame to the Silver container
gold_output_path = f"{gold_adls}earthquake_events_gold/"

In [0]:
# Append DataFrame to Gold container in Parquet format
df_with_location_sig_class.write.mode('append').parquet(gold_output_path)